# apply, map, transform

In [ ]:
import sys
sys.path.append("..")

import pandas as pd
import numpy as np
from _data import load_penguins

penguins = load_penguins()
penguins.shape

## 1. `.map()` sur une Series — remplacer des valeurs

`.map()` applique une fonction **élément par élément** sur une Series.
Cas d'usage principal : remplacer des codes par des labels, ou transformer chaque valeur indépendamment.

In [ ]:
# Remplacement via un dictionnaire — la forme la plus efficace
labels_espece = {"Adelie": "Manchot d'Adélie", "Gentoo": "Manchot papou", "Chinstrap": "Manchot à jugulaire"}
penguins["species"].map(labels_espece).head(8)

In [ ]:
# Remplacement via une fonction — pour les transformations plus complexes
penguins["body_mass_g"].map(lambda x: x / 1000 if pd.notna(x) else x).head(5)

In [ ]:
# Attention : .map() avec un dictionnaire remplace les valeurs non trouvées par NaN
labels_incomplets = {"Adelie": "Manchot d'Adélie"}  # Gentoo et Chinstrap absents
penguins["species"].map(labels_incomplets).value_counts(dropna=False)

In [ ]:
# Si vous voulez garder les valeurs non mappées, utilisez .replace() plutôt que .map()
penguins["species"].replace(labels_incomplets).value_counts()

### `.map()` remplace `applymap()` sur une DataFrame

`applymap()` a été renommé `.map()` en pandas 2.1 sur les DataFrames.
Il applique une fonction à chaque cellule individuelle.

In [ ]:
# Arrondir toutes les colonnes numériques en une passe
penguins.select_dtypes("float").map(lambda x: round(x, 1) if pd.notna(x) else x).head(4)

## 2. `.apply()` sur un DataFrame — traiter lignes ou colonnes

`.apply()` applique une fonction sur chaque **colonne** (`axis=0`) ou chaque **ligne** (`axis=1`).
C'est plus flexible que `.map()` mais aussi beaucoup plus lent — voir section 5.

In [ ]:
# axis=0 (défaut) : la fonction reçoit chaque colonne comme une Series
penguins.select_dtypes("float").apply(lambda col: col.max() - col.min())

In [ ]:
# axis=1 : la fonction reçoit chaque ligne comme une Series
# Cas typique : créer une colonne à partir de plusieurs colonnes
def categorie_morpho(row):
    """Classe l'individu selon sa taille et sa masse."""
    if pd.isna(row["body_mass_g"]) or pd.isna(row["flipper_length_mm"]):
        return np.nan
    if row["body_mass_g"] > 4500 and row["flipper_length_mm"] > 200:
        return "grand_lourd"
    elif row["body_mass_g"] < 3500:
        return "petit_léger"
    return "intermédiaire"

penguins.apply(categorie_morpho, axis=1).value_counts()

In [ ]:
# .apply() peut retourner un DataFrame (résultat avec plusieurs colonnes)
def stats_rapides(col):
    return pd.Series({"moy": col.mean(), "med": col.median(), "std": col.std()})

penguins.select_dtypes("float").apply(stats_rapides).round(1)

## 3. `.transform()` — résultat de même taille que l'entrée

`.transform()` est utilisé quand on veut que le résultat soit **aligné** sur l'index original.
Cas typique : centrer ou normaliser une variable par rapport à son groupe (vu dans le notebook 4.0).

In [ ]:
# Normalisation Z-score par espèce (mean=0, std=1 dans chaque groupe)
penguins["masse_zscore"] = (
    penguins.groupby("species")["body_mass_g"]
    .transform(lambda df_: (df_ - df_.mean()) / df_.std())
)

penguins.groupby("species")["masse_zscore"].agg(["mean", "std"]).round(10)

In [ ]:
# Imputer les NaN par la médiane du groupe — un cas classique
penguins["masse_imputee"] = (
    penguins.groupby("species")["body_mass_g"]
    .transform(lambda df_: df_.fillna(df_.median()))
)

print("NaN avant :", penguins["body_mass_g"].isna().sum())
print("NaN après :", penguins["masse_imputee"].isna().sum())

## 4. `.agg()` avec un dictionnaire

`agg()` peut être utilisé directement sur une DataFrame (pas seulement après groupby)
pour calculer des statistiques différentes par colonne.

In [ ]:
penguins.select_dtypes("float").agg({
    "bill_length_mm":    ["mean", "std"],
    "bill_depth_mm":     ["mean", "std"],
    "flipper_length_mm": ["min", "max"],
    "body_mass_g":       ["median", "mean"],
}).round(1)

## 5. Quand éviter `apply`

`.apply(axis=1)` est la fonction la plus lente de pandas.
Elle boucle en Python sur chaque ligne — exactement ce qu'on veut éviter.

**Dans 80 % des cas, il existe une alternative vectorisée.** Voici les patterns les plus courants.

In [ ]:
sample = penguins.dropna().head(3_000)

def categorie_apply(row):
    if row["body_mass_g"] > 4500 and row["flipper_length_mm"] > 200:
        return "grand_lourd"
    elif row["body_mass_g"] < 3500:
        return "petit_léger"
    return "intermédiaire"

In [ ]:
%%timeit
sample.apply(categorie_apply, axis=1)

In [ ]:
%%timeit
# Version vectorisée avec np.select — 10-50x plus rapide
conditions = [
    (sample["body_mass_g"] > 4500) & (sample["flipper_length_mm"] > 200),
    sample["body_mass_g"] < 3500,
]
choix = ["grand_lourd", "petit_léger"]
np.select(conditions, choix, default="intermédiaire")

### Alternatives vectorisées aux `apply` courants

| Besoin | `apply` (lent) | Alternative vectorisée (rapide) |
|---|---|---|
| Condition à deux cas | `apply(lambda r: A if ... else B)` | `np.where(cond, A, B)` |
| Condition à N cas | `apply(categorie, axis=1)` | `np.select([c1,c2,...], [v1,v2,...], default)` |
| Remplacer des codes | `apply(lambda x: dict[x])` | `.map(dict)` |
| Opération math sur colonnes | `apply(lambda r: r.a + r.b, axis=1)` | `df["a"] + df["b"]` |
| Manipulation de chaînes | `apply(lambda x: x.lower())` | `.str.lower()` |
| Extraction date | `apply(lambda x: x.year)` | `.dt.year` |
| Clip entre bornes | `apply(lambda x: max(0, min(x, 100)))` | `.clip(0, 100)` |

In [ ]:
# Cas où apply est vraiment nécessaire :
# une fonction qui retourne plusieurs valeurs différentes par ligne
# et qu'on ne peut pas vectoriser facilement

def analyse_bec(row):
    """Analyse composite qui combine plusieurs colonnes de façon non triviale."""
    if pd.isna(row["bill_length_mm"]) or pd.isna(row["bill_depth_mm"]):
        return pd.Series({"bec_index": np.nan, "bec_forme": "inconnu"})
    ratio = row["bill_length_mm"] / row["bill_depth_mm"]
    forme = "allongé" if ratio > 2.5 else "profond" if ratio < 1.8 else "standard"
    return pd.Series({"bec_index": round(ratio, 2), "bec_forme": forme})

penguins.join(penguins.apply(analyse_bec, axis=1))[["species", "bill_length_mm", "bill_depth_mm", "bec_index", "bec_forme"]].head(5)

## Récapitulatif

| Outil | Entrée | Sortie | Cas d'usage |
|---|---|---|---|
| `Series.map(dict)` | Valeur par valeur | Series même taille | Remplacement de codes par labels |
| `Series.map(func)` | Valeur par valeur | Series même taille | Transformation scalaire |
| `DataFrame.map(func)` | Cellule par cellule | DataFrame même taille | Transformation uniforme de toutes les cellules |
| `DataFrame.apply(func, axis=0)` | Colonne par colonne | Dépend | Stats par colonne |
| `DataFrame.apply(func, axis=1)` | Ligne par ligne | Dépend | **Lent** — à éviter, chercher vectorisé |
| `GroupBy.transform(func)` | Groupe entier | Series même taille que l'original | Normalisation, imputation par groupe |
| `np.where(cond, A, B)` | Vectorisé | Array/Series | Condition binaire rapide |
| `np.select([c1,c2], [v1,v2])` | Vectorisé | Array/Series | Conditions multiples rapides |